In [ ]:
from pathlib import Path
import yaml
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from mdx2.io import loadobj
from mdx2.report.visualization import calc_isoavg, extract_central_slice, unique_slices
from mdx2.report.map_statistics import prepare_table

with open("site_config.yaml") as f:
    config = yaml.safe_load(f)
project_directory = Path(config['project_directory'])

def subtract_isotropic_part(da, s_iso, intensity_iso):
    s = np.sqrt(da.sx**2 + da.sy**2 + da.sz**2)
    out = da.copy()
    out.data = da.data - np.interp(s, s_iso, intensity_iso)
    return out

def ingest_merged_data(merged_nxs):
    crystal   = loadobj(merged_nxs, 'crystal')
    symmetry  = loadobj(merged_nxs, 'symmetry')
    hkl_table = loadobj(merged_nxs, 'hkl_table')
    s_iso, intensity_iso = calc_isoavg(hkl_table, crystal)
    slices = []
    for slice_index, coord in unique_slices(symmetry):
        intensity = extract_central_slice(hkl_table, symmetry, crystal,
            slice_index=slice_index, slice_coordinate=coord, signal="intensity")
        slices.append({'intensity': intensity,
                       'intensity_subtracted': subtract_isotropic_part(intensity, s_iso, intensity_iso)})
    return {'isotropic': {'s': s_iso, 'intensity': intensity_iso}, 'slices': slices}

bin_width = 0.01
smax = 0.8
nbins = int(smax / bin_width)
bin_edges = np.linspace(0, bin_width*nbins, nbins + 1)

def load_merged_calc_stats(merged_nxs):
    crystal   = loadobj(merged_nxs, 'crystal')
    symmetry  = loadobj(merged_nxs, 'symmetry')
    hkl_table = loadobj(merged_nxs, 'hkl_table')
    df = prepare_table(hkl_table, crystal, symmetry)
    # exclude Bragg points
    df = df[~df["is_reflection"]]
    # compute statistics in each bin
    def rmsavg(x):
        return np.sqrt(np.mean(x * x))
    s_bins = pd.cut(df["s"], bins=bin_edges)
    stats = df.groupby(s_bins, observed=False).agg(
        s=("s", "mean"),
        intensity_iso=("intensity_iso", "mean"),
        intensity_sub_rms=("intensity_sub", rmsavg),
        intensity_error_rms=("intensity_error", rmsavg),
    )
    return stats

In [ ]:
from itertools import combinations

XTALS = ['mac1_2', 'mac1_3', 'mac1_5']

def merged_path(combo):
    return project_directory / 'chess_20250924/combos' / '+'.join(combo) / 'merged.nxs'

def combo_label(combo):
    return ' + '.join(c.replace('mac1_', 'xtal ') for c in combo)

rows = [list(combinations(XTALS, n)) for n in (1, 2, 3)]
combos = [c for row in rows for c in row]

combo_data  = {c: ingest_merged_data(merged_path(c))     for c in combos}
combo_stats = {c: load_merged_calc_stats(merged_path(c)) for c in combos}

In [ ]:
fig = plt.figure(figsize=(9, 12))
# 4 columns: three slices + a narrow colorbar column; 3 slice rows + ratio row
gs = gridspec.GridSpec(4, 4, width_ratios=[1, 1, 1, 0.06],
                       height_ratios=[1, 1, 1, 1.1], hspace=0.32, wspace=0.25)
slice_index=1
for i, row in enumerate(rows):
    cols = [1] if len(row) == 1 else range(len(row))   # center the 3-crystal panel
    for j, combo in zip(cols, row):
        ax = fig.add_subplot(gs[i, j])
        vals = combo_data[combo]
        normval = np.max(vals['isotropic']['intensity'])
        sd = vals['slices'][slice_index]['intensity_subtracted']
        s = np.sqrt(sd.sx**2 + sd.sy**2 + sd.sz**2)
        im = (sd.where(s < 0.8, np.nan) / normval).plot.pcolormesh(
            "sx", "sy", vmin=-0.12, vmax=0.12, cmap='RdBu_r', ax=ax, add_colorbar=False, rasterized=True)
        ax.set_xlim(-0.8, 0.8); ax.set_ylim(-0.8, 0.8)
        ax.set_aspect('equal', adjustable='box')
        ax.set_title(combo_label(combo))
        if j > 0 and len(combo) < 3:
            ax.set_ylabel('')
        if i < 2:
            ax.set_xlabel('')

cax = fig.add_subplot(gs[0:3, 3])
fig.colorbar(im, cax=cax)

In [ ]:
N_COLORS = {1: 'tab:blue', 2: 'tab:orange', 3: 'tab:green'}

fig, ax = plt.subplots(1, 1, figsize=(8,4))

seen = set()
for combo in combos:
    n = len(combo)
    df_ex = combo_stats[combo]
    s    = np.asarray(df_ex['s'])
    ivar = np.asarray(df_ex['intensity_sub_rms'])**2
    evar = np.asarray(df_ex['intensity_error_rms'])**2
    signal_var_est = ivar - evar
    signal_var_est[signal_var_est < 0] = np.nan   # same suppression as Fig 2
    signal_var_est[s < .04] = np.nan   # remove low-angle part (poor stats)
    irmscorr = np.sqrt(signal_var_est)            # corrected anisotropic RMS, as in Fig 2
    snr = irmscorr / np.sqrt(evar)
    ax.plot(s, snr, color=N_COLORS[n], lw=1.6, alpha=0.85,
             label=f'n = {n}' if n not in seen else None)
    seen.add(n)
ax.axhline(1, color='k', ls='--', lw=1, alpha=0.7, label='SNR=1')
ax.set_xlim(0, 0.8)
ax.set_ylim(0.5, 10)
ax.set_yscale('log')
ax.legend()
ax.set_xlabel("s (Å$^{-1}$)")
ax.set_ylabel('corrected RMS intensity / RMS error')

In [ ]:
# Dump plot data for Figure 3

import json
import xarray as xr

normval = np.max(combo_data[('mac1_2', 'mac1_3', 'mac1_5')]['isotropic']['intensity'])

slices = []

# first, stack everybody into one dataframe
for i, combos in enumerate(rows):
    for combo in combos:
        sd = combo_data[combo]['slices'][slice_index]['intensity_subtracted']
        sd = sd / normval
        sd = sd.astype('float32')
        sd = sd.rename(' + '.join(combo))
        # keep sx, sy, sz only for the last one
        if len(combo) < 3:
            sd = sd.drop_vars(('sx','sy','sz'))
        slices.append(sd)

da = xr.merge(slices)
s = np.sqrt(da.sx**2 + da.sy**2 + da.sz**2)
da = da.where(s < 0.8, np.nan)
da = da.where(da['mac1_2'].notnull(),drop=True)

In [ ]:
# now, dump into a dict and save as json

with open('figures/source_data/fig3_combo_slices_chess_20250924.json','w') as f:
    json.dump(da.to_dict(),f)

In [ ]:
!ls -lht figures/source_data/fig3_combo_slices_chess_20250924.json

In [ ]:
# dump snr curves

# first, combine into one pandas dataframe

curves = {}

# first, stack everybody into one dataframe
for i, combos in enumerate(rows):
    for combo in combos:
        df = combo_stats[combo]
        s    = df['s']
        ivar = df['intensity_sub_rms']**2
        evar = df['intensity_error_rms']**2
        signal_var_est = ivar - evar
        signal_var_est[signal_var_est < 0] = np.nan   # same suppression as Fig 2
        signal_var_est[s < .04] = np.nan   # remove low-angle part (poor stats)
        irmscorr = np.sqrt(signal_var_est)            # corrected anisotropic RMS, as in Fig 2
        snr = irmscorr / np.sqrt(evar)
        name = ' + '.join(combo)
        curves[name] = snr
        if 's' not in curves:
            curves['s'] = s

df = pd.DataFrame(curves).set_index('s')
df

In [ ]:
df.to_csv('figures/source_data/fig3_combo_snr_chess_20250924.csv')